# B3 — Step 1: Data Download & Inspection

**Block:** B3 Symbol Detection (🟥 FULL)  
**Fine-tuning step:** 1 (Get the raw data) + start of 2 (Train/test split)

This notebook does **not** download data. You upload the three datasets to Google Drive
manually; this notebook **mounts Drive, verifies each dataset is present and structurally
intact, and runs label sanity-checks** — the checks your strategy doc calls the #1 cause of
fine-tuning failure.

### What B3 uses (per block spec §3)
| Dataset | Role for B3 | Expected on Drive |
|---|---|---|
| **Kaggle P&ID Symbols** | Pretrain (synthetic, 32-class boxes) | `kaggle_pid_symbols/` |
| **Gupta PID_Dataset** | Fine-tune + **test** (real, 72 train / 20 test) | `gupta_pid/` |
| **PID2Graph OPEN100** | *(B4/B7 — not B3, but validated here for completeness)* | `pid2graph/` |

> **Data discipline (spec §9):** the Gupta 20-sheet test split is touched exactly once, at
> final scoring. This notebook only *counts and inspects* it — it never opens test labels for
> any decision.

## 0. Configuration

Set `DRIVE_ROOT` to the folder where you uploaded the datasets. Adjust the three subfolder
names if yours differ. **Nothing else in the notebook needs editing.**

In [3]:
# --- Edit these to match your Drive layout ---
DRIVE_ROOT   = "/content/drive/MyDrive/pid_project/data"  # where you uploaded the datasets
KAGGLE_DIR   = "kaggle_pid_symbols"                        # subfolder: Kaggle P&ID Symbols
GUPTA_DIR    = "gupta_pid"                                 # subfolder: Gupta PID_Dataset
PID2GRAPH_DIR= "pid2graph"                                 # subfolder: PID2Graph OPEN100

# Data version tag (spec §1 fine-tuning: record which data every experiment used)
DATA_VERSION = "data-v1"

## 1. Mount Drive & install light deps

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Only lightweight libs — no model downloads at this stage
!pip -q install pillow numpy pandas 2>/dev/null

import os, json, glob, xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
from PIL import Image

ROOT = Path(DRIVE_ROOT)
print("Drive root:", ROOT, "| exists:", ROOT.exists())

Mounted at /content/drive
Drive root: /content/drive/MyDrive/pid_project/data | exists: True


## 1b. Install full dependency stack (Checklist 0.3)

Heavy/model deps — separate from the lightweight inspection libs above since these need
the GPU runtime. No MLflow (CLAUDE.md hard rule #8) — tracking is via `results.csv` +
`experiments/stage4/v*.md`.

Covers the Stage 4 base-VLM bake-off candidates (Qwen3-VL, InternVL3, Molmo).

In [5]:
%%capture install_log
!pip install -q \
    torch \
    transformers \
    accelerate \
    vllm \
    nvidia-cuda-runtime-cu13 \
    pycocotools \
    supervision \
    kagglehub \
    kaggle \
    qwen-vl-utils \
    einops \
    timm

### Kaggle API authentication

One-time setup (see instructions below the notebook) — save your Kaggle API token to
`{DRIVE_ROOT}/kaggle_access_token.txt` (just the raw token string, no quotes), then run
this cell to install it for the session as `~/.kaggle/access_token`.

In [ ]:
import os

token_src = f"{DRIVE_ROOT}/kaggle_access_token.txt"
kaggle_dir = os.path.expanduser("~/.kaggle")
token_dst = f"{kaggle_dir}/access_token"

if os.path.exists(token_src):
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(token_src) as f:
        token = f.read().strip()
    with open(token_dst, "w") as f:
        f.write(token)
    os.chmod(token_dst, 0o600)
    print("Kaggle API token installed for this session.")
else:
    print(f"No token file found at {token_src} — see setup instructions, then re-run this cell.")

In [8]:
# --- ✓ Confirm: every package imports without error; print versions; pin requirements.txt ---
import importlib

packages = [
    "torch", "transformers", "accelerate", "vllm", "pycocotools",
    "supervision", "kagglehub", "kaggle", "qwen_vl_utils", "einops", "timm",
]

failed = []
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:15s} OK  {getattr(mod, '__version__', 'version n/a')}")
    except Exception as e:
        print(f"{pkg:15s} FAILED: {e}")
        failed.append(pkg)

if failed:
    print("\nFAILED packages:", failed)
else:
    # Pin exact versions to Drive so the environment is reproducible
    req_path = f"{DRIVE_ROOT}/requirements.txt"
    !pip freeze > "{req_path}"
    print("\nrequirements.txt saved to:", req_path)


torch           OK  2.11.0+cu128
transformers    OK  5.12.1
accelerate      OK  1.14.0
vllm            FAILED: libcudart.so.13: cannot open shared object file: No such file or directory
pycocotools     OK  version n/a
supervision     OK  0.29.1
kagglehub       OK  1.0.2
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
kaggle          FAILED: name 'exit' is not defined
qwen_vl_utils   OK  version n/a
einops          OK  0.8.2
timm            OK  1.0.27

FAILED packages: ['vllm', 'kaggle']


## 2. Presence check — are the three datasets where we expect them?

Fails loudly with a clear message if a folder is missing, so you fix the upload before wasting
a session.

In [7]:
def check_dir(name, sub):
    p = ROOT / sub
    ok = p.exists() and any(p.iterdir()) if p.exists() else False
    status = "✅" if ok else "❌"
    n_files = sum(1 for _ in p.rglob('*') if _.is_file()) if p.exists() else 0
    print(f"{status}  {name:12s} → {p}  ({n_files} files)")
    return p, ok

kaggle_p,  kaggle_ok  = check_dir("Kaggle",    KAGGLE_DIR)
gupta_p,   gupta_ok   = check_dir("Gupta",     GUPTA_DIR)
p2g_p,     p2g_ok     = check_dir("PID2Graph", PID2GRAPH_DIR)

if not (kaggle_ok and gupta_ok):
    print("\n⚠️  B3 needs Kaggle (pretrain) + Gupta (fine-tune/test). Fix uploads before continuing.")
else:
    print("\nB3-required datasets present. PID2Graph is optional for B3.")

✅  Kaggle       → /content/drive/MyDrive/pid_project/data/kaggle_pid_symbols  (36595 files)
✅  Gupta        → /content/drive/MyDrive/pid_project/data/gupta_pid  (5526 files)
✅  PID2Graph    → /content/drive/MyDrive/pid_project/data/pid2graph  (68083 files)

B3-required datasets present. PID2Graph is optional for B3.


## 3. Explore folder structure

Datasets ship with different layouts; print a shallow tree so we can see how each is organised
before hard-coding paths.

In [ ]:
def shallow_tree(root, max_depth=2, max_entries=12):
    root = Path(root)
    if not root.exists():
        print(f"(missing) {root}"); return
    for p in sorted(root.rglob('*')):
        depth = len(p.relative_to(root).parts)
        if depth > max_depth: continue
        siblings = list(p.parent.iterdir())
        if len(siblings) > max_entries and siblings.index(p) >= max_entries:
            continue
        indent = "  " * (depth - 1)
        tag = "/" if p.is_dir() else ""
        print(f"{indent}{p.name}{tag}")

for label, p, ok in [("KAGGLE", kaggle_p, kaggle_ok), ("GUPTA", gupta_p, gupta_ok), ("PID2GRAPH", p2g_p, p2g_ok)]:
    print(f"\n===== {label} =====")
    if ok: shallow_tree(p)
    else:  print("(not present)")

## 4. Kaggle P&ID Symbols — inspect (pretrain source)

Expected per spec: ~500 diagrams, ~30k tiles, **32-class** boxes, ~195k instances.
We auto-detect the annotation format (Pascal VOC XML / YOLO txt / COCO json) and report the
**class distribution** — imbalance here directly informs pretraining (strategy step 8.3).

In [ ]:
def detect_ann_format(root):
    root = Path(root)
    has = lambda ext: any(root.rglob(ext))
    fmts = []
    if has('*.xml'):  fmts.append('voc_xml')
    if any(root.rglob('*.txt')) and (has('*.jpg') or has('*.png')): fmts.append('yolo_txt?')
    if has('*.json'): fmts.append('coco_json?')
    return fmts

def count_images(root):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp','*.tif','*.tiff')
    return sum(len(list(Path(root).rglob(e))) for e in exts)

if kaggle_ok:
    print("Detected annotation formats:", detect_ann_format(kaggle_p))
    print("Image count:", count_images(kaggle_p))

    # VOC XML class histogram (Kaggle P&ID Symbols is typically VOC-style)
    class_counts = Counter()
    xmls = list(kaggle_p.rglob('*.xml'))
    for x in xmls:
        try:
            for obj in ET.parse(x).getroot().findall('.//object'):
                nm = obj.findtext('name')
                if nm: class_counts[nm] += 1
        except Exception as e:
            pass
    if class_counts:
        print(f"\nParsed {len(xmls)} XML files, {sum(class_counts.values())} instances, {len(class_counts)} classes")
        df = pd.DataFrame(class_counts.most_common(), columns=['class','count'])
        display(df)
        print("Rarest 5 classes (few-shot / Siamese candidates):")
        print(df.tail(5).to_string(index=False))
    else:
        print("\nNo VOC objects parsed — inspect the tree above; format may be YOLO/COCO. Tell me and I'll add a parser.")
else:
    print("Kaggle not present — skipping.")

## 5. Gupta PID_Dataset — inspect (fine-tune + TEST source)

Expected per spec: 92 real sheets (**72 train / 20 test**), class-agnostic "Symbol" boxes.
This is the real judging set for B3, so we verify the split counts and confirm the
class-agnostic labelling (which is what pushes the fixed-head-vs-Siamese question).

In [ ]:
if gupta_ok:
    print("Detected annotation formats:", detect_ann_format(gupta_p))
    print("Total images:", count_images(gupta_p))

    # Try to locate train/test split folders by common naming
    split_hits = defaultdict(int)
    for d in gupta_p.rglob('*'):
        if d.is_dir():
            low = d.name.lower()
            if low in ('train','training'): split_hits['train'] += count_images(d)
            if low in ('test','testing','val','valid'): split_hits[low] += count_images(d)
    print("Split folders found (by name):", dict(split_hits) or "none — split may be file-list based")

    # Class-agnostic check: collect distinct label names
    labels = Counter()
    for x in gupta_p.rglob('*.xml'):
        try:
            for obj in ET.parse(x).getroot().findall('.//object'):
                nm = obj.findtext('name')
                if nm: labels[nm] += 1
        except Exception:
            pass
    if labels:
        print(f"\nDistinct label names in Gupta: {len(labels)}")
        print(dict(labels.most_common(10)))
        if len(labels) <= 3:
            print("→ Confirms class-agnostic 'Symbol' labelling (localization GT only, not typing).")
    else:
        print("No XML labels parsed — check tree; Gupta may use a different annotation file. Report back.")
else:
    print("Gupta not present — skipping.")

## 6. Image health check

Corrupt/unreadable images and wildly inconsistent resolutions are silent fine-tuning killers.
Sample a subset from each dataset and report readability + size distribution.

In [ ]:
def image_health(root, sample=40):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp','*.tif','*.tiff')
    files = [f for e in exts for f in Path(root).rglob(e)]
    if not files:
        print("  no images found"); return
    import random; random.seed(0)
    sub = random.sample(files, min(sample, len(files)))
    sizes, bad = [], 0
    for f in sub:
        try:
            with Image.open(f) as im: sizes.append(im.size)
        except Exception:
            bad += 1
    if sizes:
        ws, hs = zip(*sizes)
        print(f"  sampled {len(sub)} | unreadable: {bad}")
        print(f"  width  min/med/max: {min(ws)}/{int(np.median(ws))}/{max(ws)}")
        print(f"  height min/med/max: {min(hs)}/{int(np.median(hs))}/{max(hs)}")

for label, p, ok in [("KAGGLE", kaggle_p, kaggle_ok), ("GUPTA", gupta_p, gupta_ok)]:
    print(f"{label}:")
    if ok: image_health(p)
    else:  print("  (not present)")
    print()

## 7. Summary manifest (for the benchmarking repo)

Writes a small JSON manifest back to Drive so `experiments/v1.md` can reference the exact
data state this run saw (spec: every experiment records its data version).

In [ ]:
manifest = {
    "data_version": DATA_VERSION,
    "drive_root": str(ROOT),
    "datasets": {
        "kaggle":    {"present": kaggle_ok, "images": count_images(kaggle_p) if kaggle_ok else 0},
        "gupta":     {"present": gupta_ok,  "images": count_images(gupta_p)  if gupta_ok  else 0},
        "pid2graph": {"present": p2g_ok,    "images": count_images(p2g_p)    if p2g_ok    else 0},
    },
}
out = ROOT / f"manifest_{DATA_VERSION}.json"
out.write_text(json.dumps(manifest, indent=2))
print("Wrote", out)
print(json.dumps(manifest, indent=2))

## What to report back to me

After running, tell me:
1. The **presence check** (§2) results — which datasets are ✅.
2. The **Kaggle class distribution** (§4) — especially the rarest classes.
3. Whether **Gupta confirmed class-agnostic** labelling (§5), and the split folder counts.
4. Any **unreadable images** or extreme size ranges (§6).

That tells us whether the data is clean enough to proceed, and it feeds the
fixed-head-vs-Siamese decision before we touch **Step 3 (baseline)**.